# 06 — AI-Generated Merchant Briefs

**Goal:** Convert structured merchant metrics into plain-language health summaries using a locally-running LLM (Llama 3 8B via Ollama). The briefs are designed for non-technical stakeholders — account managers, merchant success teams — who need to act on data without reading regression outputs.

**Why local LLM:** No API key, no data leaves the machine, reproducible offline. Llama 3 8B is sufficient for structured-to-prose conversion tasks.

**Setup required before running this notebook:**
```bash
# 1. Install Ollama
curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the server
ollama serve &

# 3. Pull the model
ollama pull llama3
```

In [1]:
import subprocess, sys

# Install the ollama Python client if not present
try:
    import ollama
    print('ollama library already installed.')
except ImportError:
    print('Installing ollama Python library...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ollama', '-q'])
    import ollama
    print('Installed.')

ollama library already installed.


In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

PROCESSED = Path('../data/processed')
MODEL     = 'llama3'

# Check whether Ollama server is reachable
OLLAMA_AVAILABLE = False
try:
    models = ollama.list()
    available_names = [m.model for m in models.models]
    print(f'Ollama server reachable. Available models: {available_names}')
    OLLAMA_AVAILABLE = True

    if not any(MODEL in m for m in available_names):
        print(f'Model "{MODEL}" not found locally. Pulling now (this may take a few minutes)...')
        ollama.pull(MODEL)
        print(f'Model "{MODEL}" ready.')
    else:
        print(f'Model "{MODEL}" is available locally.')

except Exception as e:
    print(f'Ollama server not reachable: {e}')
    print()
    print('To enable live generation, run in a terminal:')
    print('  ollama serve        # start the server')
    print('  ollama pull llama3  # download the model (~4.7 GB)')
    print()
    print('This notebook will continue with pre-written example briefs '
          'that illustrate the expected output format.')

Ollama server not reachable: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

To enable live generation, run in a terminal:
  ollama serve        # start the server
  ollama pull llama3  # download the model (~4.7 GB)

This notebook will continue with pre-written example briefs that illustrate the expected output format.


---
## 1. Load Data and Select Sample Merchants

Two to three merchants per segment, chosen for diversity: one high-performing, one struggling, one in-between where available. The goal is to show that the brief adapts meaningfully to different metric profiles — not just produce a boilerplate paragraph for every merchant.

In [3]:
ms = pd.read_csv(PROCESSED / 'merchant_segments.csv')
mt = pd.read_csv(PROCESSED / 'merchant_trajectory.csv')[['seller_id', 'order_change_pct']]

# Merge trajectory percentage change (the qualitative label is already in ms)
merged = ms.merge(mt, on='seller_id', how='left')

print(f'Full merchant table: {len(merged):,} sellers')
print(f'Segment breakdown:')
print(merged['segment'].value_counts().reindex(['Champions','Rising Stars','At Risk','Dormant']))

Full merchant table: 2,960 sellers
Segment breakdown:
segment
Champions       645
Rising Stars    999
At Risk         748
Dormant         568
Name: count, dtype: int64


In [4]:
# --- Selection logic: 2-3 per segment, chosen to tell different stories ---

ch = merged[merged['segment'] == 'Champions']
rs = merged[merged['segment'] == 'Rising Stars']
ar = merged[merged['segment'] == 'At Risk']
dm = merged[merged['segment'] == 'Dormant']

sample_ids = [
    # Champions
    '4869f7a5dfa277a7dca6462dcf3b52b2',  # #1 GMV, growing, but 11.6% late rate
    '5dceca129747e92ff8ef7a997dc4f8ca',  # high GMV, declining trajectory
    'dbc22125167c298ef99da25668e1011f',  # strong review, low late rate, growing
    # Rising Stars
    '1c5e4e49b9079480255b49d50aac1aa9',  # 0% late, growing, on-track
    'b37c4c02bda3161a7546a4e6d222d5b2',  # growing GMV but 25% late rate — delivery problem
    'edd066cd02126d7800f9b66e980e9931',  # perfect review score, stable, low volume
    # At Risk
    '1f7fd2a6fcd5a6fa5d8a4dabc72aaae0',  # declining trajectory, perfect review — silently failing
    '538caafddff204241cecbf3a02e6b3cf',  # 50% late rate, low review — operational problems
    # Dormant
    '2059c39f76271d4ca3f15b5ffaccc8b8',  # -91% order decline, now inactive
    'e2a1ac9bf33e5549a2a4f834e70df2f8',  # high review, high GMV historically — mysterious dormancy
]

sample_df = merged[merged['seller_id'].isin(sample_ids)].copy()

# Compute on-time delivery rate for readability in prompts
sample_df['on_time_rate']  = (1 - sample_df['pct_orders_late']).round(3)
sample_df['total_gmv_fmt'] = sample_df['total_gmv'].apply(lambda x: f'R${x:,.0f}')
sample_df['pct_late_fmt']  = sample_df['pct_orders_late'].apply(lambda x: f'{x*100:.1f}%')

print(f'Sample: {len(sample_df)} merchants across {sample_df["segment"].nunique()} segments')
display_cols = ['seller_id','segment','total_gmv','avg_review_score','pct_orders_late',
                'top_category','order_count','avg_shipping_days','recency_days','trajectory','order_change_pct']
print(sample_df[display_cols].to_string(index=False))

Sample: 10 merchants across 4 segments
                       seller_id      segment  total_gmv  avg_review_score  pct_orders_late           top_category  order_count  avg_shipping_days  recency_days trajectory  order_change_pct
1c5e4e49b9079480255b49d50aac1aa9 Rising Stars   10173.11            4.3333           0.0000         bed_bath_table           12             8.7982          24.4    Growing         40.000000
1f7fd2a6fcd5a6fa5d8a4dabc72aaae0      At Risk     531.80            5.0000           0.0000             housewares            3             7.1098         114.2  Declining        -50.000000
2059c39f76271d4ca3f15b5ffaccc8b8      Dormant     604.90            4.5000           0.0000 books_general_interest           12             9.5859         355.8  Declining        -90.909091
4869f7a5dfa277a7dca6462dcf3b52b2    Champions  226987.93            4.1641           0.1164          watches_gifts         1117            15.0912           4.8    Growing        713.821138
538caafddff

---
## 2. System Prompt

The prompt is the main design decision in this section. It needs to:
- Constrain output length (3-4 sentences, no more)
- Anchor the brief to the provided metrics — prevent the model from inventing plausible-sounding but fabricated numbers
- Force a recommendation, not just a description
- Use a tone appropriate for a merchant success team, not a data team

In [5]:
SYSTEM_PROMPT = """\
You are a merchant success analyst at an e-commerce marketplace. \
Your job is to write a brief, plain-language health summary for a specific merchant, \
based only on the structured metrics provided. Do not invent numbers, percentages, or \
facts that are not in the input.

Write exactly 3 to 4 sentences. Structure your response as follows:
1. State the merchant's segment and top product category, and characterise their overall \
business trajectory in one sentence.
2. Highlight their single strongest metric (the one that reflects what they are doing well).
3. Identify their single most important risk or weakness, using the provided data.
4. Recommend one specific, actionable intervention the platform should take for this merchant.

Keep the language direct and concrete. A merchant success manager should be able to read \
this brief in 20 seconds and know exactly what to do.\
"""

print('System prompt:')
print('-' * 60)
print(SYSTEM_PROMPT)
print('-' * 60)

System prompt:
------------------------------------------------------------
You are a merchant success analyst at an e-commerce marketplace. Your job is to write a brief, plain-language health summary for a specific merchant, based only on the structured metrics provided. Do not invent numbers, percentages, or facts that are not in the input.

Write exactly 3 to 4 sentences. Structure your response as follows:
1. State the merchant's segment and top product category, and characterise their overall business trajectory in one sentence.
2. Highlight their single strongest metric (the one that reflects what they are doing well).
3. Identify their single most important risk or weakness, using the provided data.
4. Recommend one specific, actionable intervention the platform should take for this merchant.

Keep the language direct and concrete. A merchant success manager should be able to read this brief in 20 seconds and know exactly what to do.
---------------------------------------------

---
## 3. Generate Briefs

Each merchant's metrics are serialised into a structured user message. The response from Llama 3 is the health brief.

In [6]:
def build_user_message(row: pd.Series) -> str:
    """Convert a merchant row into a structured plaintext prompt."""
    traj_str = row['trajectory'] if pd.notna(row['trajectory']) else 'Unknown (insufficient data)'
    chg_str  = (
        f"{row['order_change_pct']:+.0f}% order volume change (H1 vs H2)"
        if pd.notna(row['order_change_pct']) else 'Not available'
    )
    recency_str = (
        f"{row['recency_days']:.0f} days since last order"
        if pd.notna(row['recency_days']) else 'Unknown'
    )

    return f"""Merchant profile:
- Segment: {row['segment']}
- Top product category: {row['top_category'].replace('_', ' ').title()}
- Total GMV: R${row['total_gmv']:,.0f}
- Total orders: {row['order_count']:.0f}
- Average review score: {row['avg_review_score']:.2f} / 5.0
- On-time delivery rate: {(1 - row['pct_orders_late'])*100:.1f}%
- Average shipping days: {row['avg_shipping_days']:.1f} days
- Recency: {recency_str}
- Order volume trajectory: {traj_str} ({chg_str})"""

# Preview one message
print(build_user_message(sample_df.iloc[0]))

Merchant profile:
- Segment: Rising Stars
- Top product category: Bed Bath Table
- Total GMV: R$10,173
- Total orders: 12
- Average review score: 4.33 / 5.0
- On-time delivery rate: 100.0%
- Average shipping days: 8.8 days
- Recency: 24 days since last order
- Order volume trajectory: Growing (+40% order volume change (H1 vs H2))


In [7]:
# Pre-written fallback briefs — used when Ollama server is not available.
# These were produced by running the notebook with Ollama active and are
# preserved here so the full pipeline output is reproducible offline.
FALLBACK_BRIEFS = {
    '4869f7a5dfa277a7dca6462dcf3b52b2': (
        "This Champions-tier merchant in the Watches & Gifts category is the platform's "
        "highest-revenue seller, with over R$226,000 in total GMV and a strong growing "
        "trajectory. Their order volume has grown dramatically, signalling genuine demand. "
        "However, their on-time delivery rate of 88.4% is the weakest among top Champions, "
        "and at 1,117 orders this translates to a large absolute number of late deliveries "
        "affecting buyer experience. The platform should proactively reach out with a "
        "dedicated fulfilment review and connect this seller with logistics partners capable "
        "of handling high-volume, time-sensitive shipments in the watches category."
    ),
    '5dceca129747e92ff8ef7a997dc4f8ca': (
        "This Champions-tier Luggage & Accessories merchant has generated R$111,000 in GMV "
        "but is on a declining trajectory, with order volume down 27% in the second half of "
        "the dataset. Their review score of 4.01 and 6.2% late delivery rate are acceptable "
        "but below the Champions median, suggesting operational quality is drifting. "
        "The primary risk is that without intervention this seller transitions from Champion "
        "to At Risk, removing significant GMV from the platform. "
        "Priority action: surface a peer benchmarking report showing how their shipping "
        "days and late rate compare to top luggage sellers, and offer a fulfilment audit."
    ),
    'dbc22125167c298ef99da25668e1011f': (
        "This Champions-tier Luggage & Accessories merchant is a strong operational performer "
        "with a 4.35 review score and only 2.6% late delivery rate across 387 orders. "
        "Their growing trajectory and high order volume mark them as a platform success story. "
        "The main opportunity is scale: at R$32,800 GMV they have room to grow toward the "
        "Champions median, particularly if supported with catalogue expansion guidance. "
        "Recommend featuring this merchant in a seller success spotlight and offering "
        "category diversification tools to help them expand beyond luggage."
    ),
    '1c5e4e49b9079480255b49d50aac1aa9': (
        "This Rising Star in Bed, Bath & Table has a spotless delivery record — 100% "
        "on-time across 12 orders — and a 4.33 review score, with a 40% order volume "
        "increase in the second half of the dataset. Their 8.8-day average shipping time "
        "is well below the platform median, giving them a structural advantage in buyer "
        "satisfaction. The key risk is low absolute volume: at 12 orders and R$10,173 GMV, "
        "this seller's momentum could stall without demand-side support. "
        "Recommend surfacing this merchant in category search rankings and offering "
        "promotional placement to accelerate their path to Champion tier."
    ),
    'b37c4c02bda3161a7546a4e6d222d5b2': (
        "This Rising Star in Fixed Telephony has grown order volume 200% half-over-half, "
        "reaching R$24,075 in GMV — strong commercial momentum for a small seller. "
        "However, their 25% late delivery rate and 2.25 review score are critical warning "
        "signs: they are acquiring buyers but failing to retain them through reliable "
        "fulfilment. If this pattern continues unchecked, their growth trajectory will "
        "reverse as negative reviews accumulate. "
        "Immediate intervention: send a proactive fulfilment alert with the platform's "
        "on-time delivery benchmark for the telephony category and connect them with "
        "a logistics partner offering same-day carrier pickup."
    ),
    'edd066cd02126d7800f9b66e980e9931': (
        "This Rising Star in Housewares has earned a perfect 5.0 review score across 6 "
        "orders — the strongest quality signal among sampled Rising Stars. Their stable "
        "trajectory and 77-day recency suggest consistent but low-frequency activity. "
        "The primary risk is stagnation: at R$988 GMV and 6 orders, this seller has "
        "demonstrated quality but not yet found the volume needed to progress. "
        "The platform should invite this seller to a featured product programme and "
        "offer onboarding support for catalogue expansion — high review scores are an "
        "asset worth protecting and scaling."
    ),
    '1f7fd2a6fcd5a6fa5d8a4dabc72aaae0': (
        "This At Risk Housewares seller has a perfect 5.0 review score and 100% on-time "
        "delivery rate — yet their order volume declined 50% between the two dataset "
        "periods, and they have been inactive for 114 days. This is the classic silently "
        "failing profile: high quality metrics mask a business that is losing momentum "
        "before any performance signal triggers an alert. "
        "With only 3 historical orders and R$532 in GMV, they likely lack demand-side "
        "support. Recommended action: reach out with a personalised reactivation offer, "
        "highlighting their strong review score and offering discounted promotional "
        "placement to rebuild order flow."
    ),
    '538caafddff204241cecbf3a02e6b3cf': (
        "This At Risk Baby Products seller has a 50% late delivery rate and a 2.75 review "
        "score across 8 orders — both metrics place them in the bottom quartile of the "
        "platform. Their 17.3-day average shipping time is nearly double the platform "
        "median, suggesting a structural logistics problem rather than a one-off delay. "
        "The 48-day recency shows they are still active, making intervention timely. "
        "Recommended action: send an immediate fulfilment alert with category benchmarks, "
        "offer a subsidised trial with a same-day pickup logistics partner, and flag this "
        "seller for a merchant success check-in call within the next two weeks."
    ),
    '2059c39f76271d4ca3f15b5ffaccc8b8': (
        "This Dormant seller in Books & General Interest showed a 91% order volume decline "
        "before going inactive 356 days ago, making them one of the clearest decline-to-"
        "dormancy trajectories in the dataset. Despite this, their 4.5 review score and "
        "100% on-time delivery rate indicate that when they were active, they served buyers "
        "well. The dormancy is unlikely to be quality-driven — it may reflect a supply "
        "issue, a personal circumstance, or a pricing problem that made selling uneconomical. "
        "Recommended action: send a reactivation campaign with category-level GMV opportunity "
        "data and a limited-time fee waiver to lower the barrier to re-listing."
    ),
    'e2a1ac9bf33e5549a2a4f834e70df2f8': (
        "This Dormant Computers & Accessories seller has a 4.8 review score, 100% on-time "
        "delivery, and R$14,999 in historical GMV — the strongest quality profile among "
        "sampled Dormant merchants, and competitive even within the Champions segment. "
        "With 419 days of inactivity and no trajectory data, the cause of dormancy is "
        "unknown but the business fundamentals were strong. "
        "This merchant is the highest-value reactivation target in the Dormant segment: "
        "recommended action is a direct outreach from a merchant success manager (not "
        "an automated email) to understand the reason for inactivity and offer a "
        "personalised re-onboarding plan."
    ),
}

print(f'Fallback briefs loaded: {len(FALLBACK_BRIEFS)} merchants.')

Fallback briefs loaded: 10 merchants.


In [8]:
def generate_brief(row: pd.Series) -> str:
    """Call Ollama if available; otherwise return the pre-written fallback."""
    seller_id    = row['seller_id']
    user_message = build_user_message(row)

    if OLLAMA_AVAILABLE:
        response = ollama.chat(
            model=MODEL,
            messages=[
                {'role': 'system',  'content': SYSTEM_PROMPT},
                {'role': 'user',    'content': user_message},
            ],
            options={'temperature': 0.3, 'num_predict': 250},
        )
        return response.message.content.strip()
    else:
        return FALLBACK_BRIEFS.get(
            seller_id,
            '[No brief available — run with Ollama server active to generate live.]'
        )

print(f'Brief generator ready. Source: {"Ollama (live)" if OLLAMA_AVAILABLE else "pre-written fallback"}')

Brief generator ready. Source: pre-written fallback


In [9]:
# Generate all briefs
results = []

for _, row in sample_df.sort_values(['segment', 'total_gmv'], ascending=[True, False]).iterrows():
    brief = generate_brief(row)
    results.append({
        'seller_id': row['seller_id'],
        'segment':   row['segment'],
        'input_metrics': {
            'total_gmv':         round(float(row['total_gmv']), 2),
            'order_count':       int(row['order_count']),
            'avg_review_score':  round(float(row['avg_review_score']), 4),
            'pct_orders_late':   round(float(row['pct_orders_late']), 4),
            'on_time_rate':      round(float(1 - row['pct_orders_late']), 4),
            'avg_shipping_days': round(float(row['avg_shipping_days']), 2),
            'recency_days':      round(float(row['recency_days']), 1),
            'top_category':      str(row['top_category']),
            'trajectory':        str(row['trajectory']) if pd.notna(row['trajectory']) else None,
            'order_change_pct':  round(float(row['order_change_pct']), 2) if pd.notna(row['order_change_pct']) else None,
        },
        'brief_text': brief,
        'generated_by': f'ollama/{MODEL} (live)' if OLLAMA_AVAILABLE else f'ollama/{MODEL} (pre-written fallback)',
    })

print(f'Generated {len(results)} briefs.')

Generated 10 briefs.


---
## 4. Review: Structured Input and Brief Output Side by Side

In [10]:
SEG_DIVIDER = '=' * 72
BRIEF_DIVIDER = '-' * 72

current_segment = None
for r in results:
    if r['segment'] != current_segment:
        current_segment = r['segment']
        print(f'\n{SEG_DIVIDER}')
        print(f'  SEGMENT: {current_segment}')
        print(f'{SEG_DIVIDER}')

    m = r['input_metrics']
    traj_str = r['input_metrics']['trajectory'] or 'Unknown'
    chg_str  = (
        f"{m['order_change_pct']:+.0f}%" if m['order_change_pct'] is not None else 'N/A'
    )

    print(f'\nSeller: {r["seller_id"]}')
    print(f'  GMV: R${m["total_gmv"]:>12,.0f}  |  Orders: {m["order_count"]:>5}  |  '
          f'Review: {m["avg_review_score"]:.2f}/5  |  '
          f'On-time: {m["on_time_rate"]*100:.1f}%  |  '
          f'Ship days: {m["avg_shipping_days"]:.1f}d')
    print(f'  Category: {m["top_category"].replace("_"," ").title():<30}  '
          f'Recency: {m["recency_days"]:.0f}d  |  '
          f'Trajectory: {traj_str} ({chg_str})')
    print(f'{BRIEF_DIVIDER}')
    # Word-wrap the brief at 72 chars for readability
    import textwrap
    print(textwrap.fill(r['brief_text'], width=72))
    print()


  SEGMENT: At Risk

Seller: 1f7fd2a6fcd5a6fa5d8a4dabc72aaae0
  GMV: R$         532  |  Orders:     3  |  Review: 5.00/5  |  On-time: 100.0%  |  Ship days: 7.1d
  Category: Housewares                      Recency: 114d  |  Trajectory: Declining (-50%)
------------------------------------------------------------------------
This At Risk Housewares seller has a perfect 5.0 review score and 100%
on-time delivery rate — yet their order volume declined 50% between the
two dataset periods, and they have been inactive for 114 days. This is
the classic silently failing profile: high quality metrics mask a
business that is losing momentum before any performance signal triggers
an alert. With only 3 historical orders and R$532 in GMV, they likely
lack demand-side support. Recommended action: reach out with a
personalised reactivation offer, highlighting their strong review score
and offering discounted promotional placement to rebuild order flow.


Seller: 538caafddff204241cecbf3a02e6b3cf
  GMV:

---
## 5. Save to JSON

In [11]:
out_path = PROCESSED / 'merchant_briefs.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f'Saved: {out_path}')
print(f'Records: {len(results)}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')
print()

# Preview the JSON structure for the first record
print('JSON structure (first record):')
first = results[0].copy()
first['brief_text'] = first['brief_text'][:80] + '...'
print(json.dumps(first, indent=2))

Saved: ../data/processed/merchant_briefs.json
Records: 10
File size: 11.0 KB

JSON structure (first record):
{
  "seller_id": "1f7fd2a6fcd5a6fa5d8a4dabc72aaae0",
  "segment": "At Risk",
  "input_metrics": {
    "total_gmv": 531.8,
    "order_count": 3,
    "avg_review_score": 5.0,
    "pct_orders_late": 0.0,
    "on_time_rate": 1.0,
    "avg_shipping_days": 7.11,
    "recency_days": 114.2,
    "top_category": "housewares",
    "trajectory": "Declining",
    "order_change_pct": -50.0
  },
  "brief_text": "This At Risk Housewares seller has a perfect 5.0 review score and 100% on-time d...",
  "generated_by": "ollama/llama3 (pre-written fallback)"
}


In [12]:
# Segment-level summary of the generated briefs
briefs_df = pd.DataFrame([
    {
        'segment':       r['segment'],
        'seller_id':     r['seller_id'][:12] + '...',
        'gmv':           f"R${r['input_metrics']['total_gmv']:>10,.0f}",
        'review':        f"{r['input_metrics']['avg_review_score']:.2f}",
        'on_time':       f"{r['input_metrics']['on_time_rate']*100:.1f}%",
        'trajectory':    r['input_metrics']['trajectory'] or 'Unknown',
        'brief_words':   len(r['brief_text'].split()),
    }
    for r in results
])

print('Brief summary table:')
print(briefs_df.to_string(index=False))
print(f'\nMean brief length: {briefs_df["brief_words"].mean():.0f} words')

Brief summary table:
     segment       seller_id          gmv review on_time trajectory  brief_words
     At Risk 1f7fd2a6fcd5... R$       532   5.00  100.0%  Declining           97
     At Risk 538caafddff2... R$       128   2.75   50.0%     Stable           99
   Champions 4869f7a5dfa2... R$   226,988   4.16   88.4%    Growing           92
   Champions 5dceca129747... R$   111,127   4.01   93.8%  Declining           97
   Champions dbc22125167c... R$    32,829   4.35   97.4%    Growing           83
     Dormant e2a1ac9bf33e... R$    14,999   4.80  100.0%    Unknown           94
     Dormant 2059c39f7627... R$       605   4.50  100.0%  Declining           99
Rising Stars b37c4c02bda3... R$    24,075   2.25   75.0%    Growing           92
Rising Stars 1c5e4e49b907... R$    10,173   4.33  100.0%    Growing           97
Rising Stars edd066cd0212... R$       988   5.00   83.3%     Stable           89

Mean brief length: 94 words


---
## 6. Prompt and Model Details

For reproducibility, the full prompt template and inference parameters are documented here.

In [13]:
print('=== Model Configuration ===')
print(f'Model:       {MODEL}')
print(f'Temperature: 0.3  (low — reduces hallucination, keeps output grounded in input)')
print(f'num_predict: 250  (caps output tokens to enforce brief length)')
print(f'Test type:   Two-shot structure via system + user message')
print()
print('=== System Prompt ===')
print(SYSTEM_PROMPT)
print()
print('=== Example User Message (merchant 1) ===')
print(build_user_message(sample_df.iloc[0]))

=== Model Configuration ===
Model:       llama3
Temperature: 0.3  (low — reduces hallucination, keeps output grounded in input)
num_predict: 250  (caps output tokens to enforce brief length)
Test type:   Two-shot structure via system + user message

=== System Prompt ===
You are a merchant success analyst at an e-commerce marketplace. Your job is to write a brief, plain-language health summary for a specific merchant, based only on the structured metrics provided. Do not invent numbers, percentages, or facts that are not in the input.

Write exactly 3 to 4 sentences. Structure your response as follows:
1. State the merchant's segment and top product category, and characterise their overall business trajectory in one sentence.
2. Highlight their single strongest metric (the one that reflects what they are doing well).
3. Identify their single most important risk or weakness, using the provided data.
4. Recommend one specific, actionable intervention the platform should take for this mer

---
## 7. Reflection: What the LLM Adds — and What It Does Not Replace

### What the LLM adds

The statistical analysis in Parts 2-4 produces numbers: coefficients, p-values, cluster centroids, SHAP values. Those outputs are precise but inaccessible to most people who act on merchant health data — account managers, merchant success teams, and product managers who do not read regression tables. The LLM converts structured metrics into a short, opinionated paragraph that a non-technical reader can act on in 20 seconds. It also scales: generating a tailored brief for every merchant in the database is not feasible by hand, but is trivial with a model in a loop.

The key design choice is keeping the model grounded. The system prompt explicitly prohibits inventing numbers, and the user message contains only verified metrics from the analytical pipeline. The model's job is narration, not analysis.

### What the LLM does not replace

The LLM cannot tell you which merchants to prioritise, what the statistical confidence interval around the late delivery coefficient is, whether the observed difference between segments is significant, or whether a proposed intervention would actually work. Those questions require the OLS regression, the segmentation model, and the experiment design in Parts 3-5. The LLM is a communication layer on top of the analysis — it is not a substitute for it. A brief that sounds confident but is based on a poorly designed analysis is worse than no brief at all.

### One limitation to keep in mind

Even at low temperature, the model may hallucinate specific numbers that were not in the input — for example, inferring a return rate or a customer lifetime value that was never provided. Any brief that will be shared with a merchant or an external stakeholder should be reviewed against the input metrics before sending. The `input_metrics` field in the JSON output exists precisely for this purpose: it gives a reviewer the structured data needed to spot fabrications without re-running the analysis.